# TT-rank / unfolding: 一般d階テンソルでの確認（src版）

元Notebook [`01_tt_rank_unfolding.ipynb`](../00_fundamentals/01_tt_rank_unfolding.ipynb) では、`tt_unfold` / `tt_svd_exact` / `tt_reconstruct` をNotebook内で定義しました。

このsrc版では `nn_compression.compression` のAPIを使い、同じ照合実験を行います。

## ゴール
- cut unfoldingを確認する
- 各cutのrankを確認する
- 打ち切りなしTT-SVDを実行する
- coreから読めるTT-rankとunfolding rankを比較する
- 完全再構成と保存量を確認する

> `torch.linalg.matrix_rank` はしきい値に基づく数値rankを返す。

## 1. 実験対象

$X \in \mathbb{R}^{2 \times 3 \times 2 \times 2}$ を使う。

In [1]:
import torch

from nn_compression.compression import (
    tt_unfold,
    tt_svd_exact,
    tt_reconstruct,
    tt_num_parameters,
)
from nn_compression.metrics import relative_frobenius_error

torch.set_default_dtype(torch.float64)
torch.manual_seed(1)

X = torch.randn(2, 3, 2, 2)
print(f"X.shape = {tuple(X.shape)}")

X.shape = (2, 3, 2, 2)


## 2. cut unfolding

切断 $1{:}k \mid k{+}1{:}d$ に対応する行列を `tt_unfold(X, k)` で作る。

4階ならshapeは
- $k = 1$: $(n_1,\ n_2 n_3 n_4)$
- $k = 2$: $(n_1 n_2,\ n_3 n_4)$
- $k = 3$: $(n_1 n_2 n_3,\ n_4)$

## 3. unfolding rank

$$
r_k = \operatorname{rank}\left( X^{\langle k \rangle} \right)
$$

各cutのshapeとrankを求め、`exact_ranks` に保存する。

In [2]:
exact_ranks = []

for k in range(1, X.ndim):
    unfolded = tt_unfold(X, k)
    r = torch.linalg.matrix_rank(unfolded)
    exact_ranks.append({
        "cut": k,
        "shape": tuple(unfolded.shape),
        "rank": r,
    })

for item in exact_ranks:
    print(f"cut={item['cut']}, shape={item['shape']}, rank={int(item['rank'].item())}")

cut=1, shape=(2, 12), rank=2
cut=2, shape=(6, 4), rank=4
cut=3, shape=(12, 2), rank=2


## 4. 一般d階のTT-SVD

```python
cores = tt_svd_exact(X)
```

各coreは

$$
G^{(k)} \in \mathbb{R}^{r_{k-1} \times n_k \times r_k}
$$

のshapeを持つ。

## 5. TT-rankの照合

- core shapeを表示する
- `cores[:-1]` の右bond dimensionからTT-rank列を取り出す
- `exact_ranks` と比較する

In [3]:
cores = tt_svd_exact(X)

for i, core in enumerate(cores, start=1):
    print(f"G{i}.shape = {tuple(core.shape)}")

core_ranks = [core.shape[2] for core in cores[:-1]]
unfold_ranks = [int(item['rank'].item()) for item in exact_ranks]
print("core_ranks:", core_ranks)
print("unfold_ranks:", unfold_ranks)
print("match:", core_ranks == unfold_ranks)

G1.shape = (1, 2, 2)
G2.shape = (2, 3, 4)
G3.shape = (4, 2, 2)
G4.shape = (2, 2, 1)
core_ranks: [2, 4, 2]
unfold_ranks: [2, 4, 2]
match: True


## 6. 完全再構成と保存量

- relative Frobenius error
- dense要素数
- TT core総要素数（`tt_num_parameters`）

In [4]:
X_hat = tt_reconstruct(cores)
rel_error = relative_frobenius_error(X, X_hat)

dense_params = X.numel()
tt_params = tt_num_parameters(cores)

print(f"X_hat.shape = {tuple(X_hat.shape)}")
print(f"relative Frobenius error = {rel_error.item():.3e}")
print(f"dense params = {dense_params}")
print(f"TT params = {tt_params}")

X_hat.shape = (2, 3, 2, 2)
relative Frobenius error = 8.353e-16
dense params = 24
TT params = 48
